# CatBoost Native-Categorical Challenger

CatBoost is tested because IEEE-CIS contains high-cardinality categories that frequency encoding may compress too aggressively. The candidate uses raw predictors plus existing past-only historical amount features. No graph, recency, count, velocity, target encoding, calibration, or threshold optimization is included.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
)
from fraud_detection.models import (
    evaluate_catboost_fold,
    evaluate_probabilities,
    make_expanding_time_folds,
    prepare_catboost_data,
    summarize_catboost_cv,
)

notebook_started = time.perf_counter()

## Native categorical and temporal strategy

Object-like columns plus card1-card6 and addr1-addr2 are native categories. Missing categories use __MISSING__; numeric NaNs remain native. Three expanding TRAIN-only folds use complete timestamps. Every fold rebuilds its historical amount state and CatBoost learns category statistics from fit rows only.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validation = split.validation
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_rows = []
for fold_number, fold in enumerate(folds, start=1):
    fit_period = train.iloc[fold.train_indices]
    evaluation_period = train.iloc[fold.evaluation_indices]
    fold_rows.append(
        {
            "Fold": fold_number,
            "Fit Rows": len(fit_period),
            "Evaluation Rows": len(evaluation_period),
            "Fit DT Max": fit_period["TransactionDT"].max(),
            "Evaluation DT Min": evaluation_period["TransactionDT"].min(),
            "Evaluation DT Max": evaluation_period["TransactionDT"].max(),
        }
    )
fold_design = pd.DataFrame(fold_rows)
display(fold_design)
print("External validation status: reserved until CV selection is complete")
print("Final holdout status: not assigned or accessed")

common_parameters = {
    "loss_function": "Logloss",
    "eval_metric": "PRAUC:type=Classic",
    "auto_class_weights": "Balanced",
    "random_seed": 42,
    "allow_writing_files": False,
    "thread_count": -1,
    "task_type": "CPU",
    "border_count": 64,
    "max_ctr_complexity": 1,
    "one_hot_max_size": 10,
}
candidate_parameters = [
    dict(
        iterations=400,
        depth=6,
        learning_rate=0.08,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        bagging_temperature=1.0,
    ),
    dict(
        iterations=600,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=5.0,
        random_strength=0.5,
        bagging_temperature=0.5,
    ),
    dict(
        iterations=500,
        depth=8,
        learning_rate=0.06,
        l2_leaf_reg=8.0,
        random_strength=1.0,
        bagging_temperature=1.0,
    ),
    dict(
        iterations=700,
        depth=6,
        learning_rate=0.04,
        l2_leaf_reg=10.0,
        random_strength=0.5,
        bagging_temperature=0.0,
    ),
]
candidate_parameters = [{**common_parameters, **item} for item in candidate_parameters]
display(pd.DataFrame(candidate_parameters))

,Fold,Fit Rows,Evaluation Rows,Fit DT Max,Evaluation DT Min,Evaluation DT Max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


External validation status: reserved until CV selection is complete
Final holdout status: not assigned or accessed


,loss_function,eval_metric,auto_class_weights,random_seed,allow_writing_files,thread_count,task_type,border_count,max_ctr_complexity,one_hot_max_size,iterations,depth,learning_rate,l2_leaf_reg,random_strength,bagging_temperature
0,Logloss,PRAUC:type=Classic,Balanced,42,False,-1,CPU,64,1,10,400,6,0.08,3.0,1.0,1.0
1,Logloss,PRAUC:type=Classic,Balanced,42,False,-1,CPU,64,1,10,600,7,0.05,5.0,0.5,0.5
2,Logloss,PRAUC:type=Classic,Balanced,42,False,-1,CPU,64,1,10,500,8,0.06,8.0,1.0,1.0
3,Logloss,PRAUC:type=Classic,Balanced,42,False,-1,CPU,64,1,10,700,6,0.04,10.0,0.5,0.0


In [3]:
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-catboost-challenge")
fold_evaluations = [[] for _ in candidate_parameters]
categorical_feature_count = 0
feature_count = 0
maximum_prepared_mb = 0.0
cv_started = time.perf_counter()

for fold_number, fold in enumerate(folds, start=1):
    fold_train = train.iloc[fold.train_indices]
    fold_evaluation = train.iloc[fold.evaluation_indices]
    prepared = prepare_catboost_data(fold_train, fold_evaluation)
    categorical_feature_count = len(prepared.categorical_indices)
    feature_count = prepared.feature_count
    prepared_mb = (
        prepared.train.memory_usage(deep=True).sum()
        + prepared.evaluation.memory_usage(deep=True).sum()
    ) / 1024**2
    maximum_prepared_mb = max(maximum_prepared_mb, prepared_mb)
    y_fold_train = fold_train["isFraud"].astype("int8")
    y_fold_evaluation = fold_evaluation["isFraud"].astype("int8")
    for configuration, parameters in enumerate(candidate_parameters):
        evaluation = evaluate_catboost_fold(
            parameters,
            prepared,
            y_fold_train,
            y_fold_evaluation,
            early_stopping_rounds=50,
        )
        fold_evaluations[configuration].append(evaluation)
        print(
            f"fold={fold_number} config={configuration + 1} "
            f"PR-AUC={evaluation.pr_auc:.6f} iteration={evaluation.best_iteration}"
        )
    del prepared, fold_train, fold_evaluation, y_fold_train, y_fold_evaluation
    gc.collect()

summaries = [
    summarize_catboost_cv(parameters, evaluations)
    for parameters, evaluations in zip(candidate_parameters, fold_evaluations, strict=True)
]
cv_rows = []
for configuration, summary in enumerate(summaries, start=1):
    with mlflow.start_run(run_name=f"catboost-cv-config-{configuration}") as run:
        mlflow.set_tags(
            {
                "stage": "model-challenger",
                "dataset": "ieee-cis",
                "selection": "temporal-cv",
                "model": "catboost",
            }
        )
        mlflow.log_params(
            {
                "model": "catboost",
                "feature_set": "raw_plus_historical_amounts",
                "categorical_feature_count": categorical_feature_count,
                "configuration": configuration,
                **summary.parameters,
            }
        )
        metrics = {
            "mean_cv_pr_auc": summary.mean_pr_auc,
            "std_cv_pr_auc": summary.std_pr_auc,
            "mean_cv_roc_auc": summary.mean_roc_auc,
        }
        for fold_index, (pr_auc, roc_auc, iteration) in enumerate(
            zip(summary.fold_pr_auc, summary.fold_roc_auc, summary.best_iterations, strict=True),
            start=1,
        ):
            metrics[f"fold_{fold_index}_pr_auc"] = pr_auc
            metrics[f"fold_{fold_index}_roc_auc"] = roc_auc
            metrics[f"fold_{fold_index}_best_iteration"] = iteration
        mlflow.log_metrics(metrics)
        cv_rows.append(
            {
                "Configuration": configuration,
                "Fold PR-AUC": summary.fold_pr_auc,
                "Mean PR-AUC": summary.mean_pr_auc,
                "Std PR-AUC": summary.std_pr_auc,
                "Mean ROC-AUC": summary.mean_roc_auc,
                "Best Iterations": summary.best_iterations,
                "MLflow Run ID": run.info.run_id,
            }
        )
cv_results = (
    pd.DataFrame(cv_rows).sort_values("Mean PR-AUC", ascending=False).reset_index(drop=True)
)
cv_seconds = time.perf_counter() - cv_started
display(cv_results)

2026/10/06 16:53:25 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-catboost-challenge' does not exist. Creating a new experiment.


fold=1 config=1 PR-AUC=0.547806 iteration=376


fold=1 config=2 PR-AUC=0.565204 iteration=511


fold=1 config=3 PR-AUC=0.569977 iteration=455


fold=1 config=4 PR-AUC=0.553634 iteration=698


...

fold=3 config=1 PR-AUC=0.610469 iteration=399


fold=3 config=2 PR-AUC=0.618580 iteration=600


fold=3 config=3 PR-AUC=0.625724 iteration=494


fold=3 config=4 PR-AUC=0.601654 iteration=556


,Configuration,Fold PR-AUC,Mean PR-AUC,Std PR-AUC,Mean ROC-AUC,Best Iterations,MLflow Run ID
0,3,"(0.5699774190938638, 0.5497721445100408, 0.625...",0.581825,0.032119,0.914958,"(455, 452, 494)",3c56b65db5e246009479a099497039c9
1,2,"(0.5652044947567489, 0.5460495637926684, 0.618...",0.576611,0.030689,0.915253,"(511, 574, 600)",fc0af5bec59145d1afb29a60510b222b
2,4,"(0.553634398299632, 0.534687454387589, 0.60165...",0.563325,0.028185,0.910760,"(698, 699, 556)",551a47d0166e4df0852253983c24ebe3
3,1,"(0.5478057626784236, 0.5293719485822028, 0.610...",0.562549,0.034710,0.910348,"(376, 390, 399)",c9d2b71885734542afc6187fa07ff12b


## One-time external validation

The winner is now fixed by mean TRAIN-CV PR-AUC. Only at this point is external VALIDATION prepared and scored. It did not influence categorical handling, early stopping, ranking, or iteration selection.

In [4]:
best_row = cv_results.iloc[0]
best_summary = summaries[int(best_row["Configuration"]) - 1]
final_parameters = dict(best_summary.parameters)
final_parameters["iterations"] = max(1, int(round(float(np.median(best_summary.best_iterations)))))
prepared_final = prepare_catboost_data(train, validation)
final_train_pool = Pool(
    prepared_final.train,
    train["isFraud"].astype("int8"),
    cat_features=prepared_final.categorical_indices,
)
final_validation_pool = Pool(
    prepared_final.evaluation,
    validation["isFraud"].astype("int8"),
    cat_features=prepared_final.categorical_indices,
)
final_model = CatBoostClassifier(**final_parameters)
final_model.fit(final_train_pool, verbose=False)
validation_probabilities = final_model.predict_proba(final_validation_pool)[:, 1]
validation_metrics = evaluate_probabilities(
    validation["isFraud"].astype("int8"), validation_probabilities
)
with mlflow.start_run(run_name="catboost-final-refit") as run:
    mlflow.set_tags(
        {
            "stage": "model-challenger",
            "dataset": "ieee-cis",
            "selection": "temporal-cv",
            "model": "catboost",
            "final_refit": "true",
        }
    )
    mlflow.log_params(
        {
            "model": "catboost",
            "feature_set": "raw_plus_historical_amounts",
            "feature_count": prepared_final.feature_count,
            "categorical_feature_count": len(prepared_final.categorical_indices),
            **final_parameters,
        }
    )
    mlflow.log_metrics(
        {
            "validation_pr_auc": validation_metrics.pr_auc,
            "validation_roc_auc": validation_metrics.roc_auc,
            "validation_precision_at_0_50": validation_metrics.precision,
            "validation_recall_at_0_50": validation_metrics.recall,
            "validation_f1_at_0_50": validation_metrics.f1,
            "feature_count": prepared_final.feature_count,
            "categorical_feature_count": len(prepared_final.categorical_indices),
        }
    )
    final_run_id = run.info.run_id
print(f"Final CatBoost iterations: {final_parameters['iterations']}")
print(f"Validation PR-AUC: {validation_metrics.pr_auc:.6f}")
print(f"Final MLflow run: {final_run_id}")

Final CatBoost iterations: 455
Validation PR-AUC: 0.566402
Final MLflow run: ef99fdf63da74675a299334d92284ce1


In [5]:
lightgbm_reference = {
    "Model": "LightGBM reference",
    "PR-AUC": 0.593452,
    "ROC-AUC": 0.927279,
    "Precision": 0.470946,
    "Recall": 0.610125,
    "F1": 0.531577,
    "CV Mean PR-AUC": 0.592712,
    "CV Std PR-AUC": 0.031409,
}
catboost_result = {
    "Model": "CatBoost challenger",
    "PR-AUC": validation_metrics.pr_auc,
    "ROC-AUC": validation_metrics.roc_auc,
    "Precision": validation_metrics.precision,
    "Recall": validation_metrics.recall,
    "F1": validation_metrics.f1,
    "CV Mean PR-AUC": best_summary.mean_pr_auc,
    "CV Std PR-AUC": best_summary.std_pr_auc,
}
comparison = pd.DataFrame([lightgbm_reference, catboost_result]).sort_values(
    "PR-AUC", ascending=False
)
display(comparison)
absolute_change = validation_metrics.pr_auc - lightgbm_reference["PR-AUC"]
relative_change = absolute_change / lightgbm_reference["PR-AUC"]
new_preferred = validation_metrics.pr_auc > lightgbm_reference["PR-AUC"]
conclusion = (
    "CatBoost becomes the preferred model by validation PR-AUC."
    if new_preferred
    else "CatBoost does not beat LightGBM; LightGBM remains preferred."
)


def cv_markdown(frame):
    rows = [
        "| Config | Fold PR-AUC | Mean | Std | Mean ROC-AUC | Iterations |",
        "| ---: | --- | ---: | ---: | ---: | --- |",
    ]
    for _, row in frame.iterrows():
        folds_text = " / ".join(f"{value:.6f}" for value in row["Fold PR-AUC"])
        iterations = " / ".join(str(value) for value in row["Best Iterations"])
        rows.append(
            f"| {int(row['Configuration'])} | {folds_text} | {row['Mean PR-AUC']:.6f} | "
            f"{row['Std PR-AUC']:.6f} | {row['Mean ROC-AUC']:.6f} | {iterations} |"
        )
    return "\n".join(rows)


parameter_lines = "\n".join(
    f"{index}. {json.dumps(parameters, sort_keys=True)}"
    for index, parameters in enumerate(candidate_parameters, start=1)
)
report = f"""# CatBoost Native-Categorical Challenger

## Why CatBoost and feature strategy

CatBoost preserves category identity and learns ordered categorical statistics. The candidate
uses {prepared_final.feature_count} raw-plus-historical-amount features, including
{len(prepared_final.categorical_indices)} native categorical features. Missing categories use
__MISSING__; numeric missing values remain native. isFraud and TransactionID are excluded.
Class imbalance uses auto_class_weights=Balanced consistently.

## Temporal CV and configurations

Three expanding TRAIN-only folds preserve timestamps. Each fold rebuilds historical amount
state. Evaluation labels are used only for fold-local early stopping and metrics. External
VALIDATION is untouched until winner selection.

{parameter_lines}

## TRAIN-only CV results

{cv_markdown(cv_results)}

- Best configuration: {int(best_row["Configuration"])}
- Best parameters: {json.dumps(best_summary.parameters, sort_keys=True)}
- Final full-TRAIN iterations: {final_parameters["iterations"]}

## External validation

- PR-AUC: {validation_metrics.pr_auc:.6f}
- ROC-AUC: {validation_metrics.roc_auc:.6f}
- Precision at 0.50: {validation_metrics.precision:.6f}
- Recall at 0.50: {validation_metrics.recall:.6f}
- F1 at 0.50: {validation_metrics.f1:.6f}
- Confusion matrix: {validation_metrics.confusion_matrix}

## LightGBM comparison and conclusion

- LightGBM reference PR-AUC: {lightgbm_reference["PR-AUC"]:.6f}
- CatBoost absolute change: {absolute_change:+.6f}
- CatBoost relative change: {relative_change:+.2%}
- {conclusion}

CatBoost avoids frequency encoding but requires categorical schema handling and heavier CPU
inference. LightGBM uses a simpler all-numeric matrix.

## MLflow

- Experiment: fraud-detection-catboost-challenge
- Winning CV run: {best_row["MLflow Run ID"]}
- Final-refit run: {final_run_id}

## Runtime, memory, and limitations

- CV workflow: {cv_seconds:.1f} seconds
- Total experiment before report write: {time.perf_counter() - notebook_started:.1f} seconds
- Largest prepared fold pair: {maximum_prepared_mb:.1f} MiB
- Only four curated configurations were tested. Threshold 0.50 is a reference only.
- No calibration, SHAP, rules, ensembling, or holdout evaluation is performed.
"""
report_path = PROJECT_ROOT / "reports" / "catboost_challenge_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Absolute PR-AUC change: {absolute_change:+.6f}")
print(f"Relative PR-AUC change: {relative_change:+.2%}")
print(conclusion)
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")

,Model,PR-AUC,ROC-AUC,Precision,Recall,F1,CV Mean PR-AUC,CV Std PR-AUC
0,LightGBM reference,0.593452,0.927279,0.470946,0.610125,0.531577,0.592712,0.031409
1,CatBoost challenger,0.566402,0.918453,0.264790,0.726824,0.388167,0.581825,0.032119


Absolute PR-AUC change: -0.027050
Relative PR-AUC change: -4.56%
CatBoost does not beat LightGBM; LightGBM remains preferred.
Report: reports\catboost_challenge_summary.md


## Scope boundary

This notebook stops after the CatBoost challenger comparison. It does not implement ensembling, target encoding, time weighting, calibration, threshold optimization, rules, SHAP, holdout evaluation, retraining, serving, or deployment.